In [1]:
import pandas as pd
import requests
import io
import os
from datetime import datetime


# paths
save_dir = "/lakehouse/default/Files/data/raw/nse_block_inc"
log_file = "/lakehouse/default/Files/data/raw/logs/pipeline_log.csv"

os.makedirs(save_dir, exist_ok=True)
os.makedirs("/lakehouse/default/Files/data/raw/logs", exist_ok=True)

# today date
# today = datetime.today().strftime("%d-%m-%Y")
today = "29-05-2026"
file_path = os.path.join(
    save_dir,
    f"blockdeals_{today}.csv"
)

try:

    # session + headers
    session = requests.Session()

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept-Encoding": "gzip, deflate"
    }

    # get cookies
    session.get(
        "https://www.nseindia.com",
        headers=headers
    )

    # API URL
    url = (
        "https://www.nseindia.com/api/"
        "historicalOR/bulk-block-short-deals"
        f"?optionType=block_deals"
        f"&from={today}"
        f"&to={today}"
        f"&csv=true"
    )

    response = session.get(
        url,
        headers=headers
    )

    # read csv
    df = pd.read_csv(
        io.StringIO(response.text)
    )

    # clean columns
    df.columns = (
        df.columns
        .str.replace("ï»¿", "", regex=False)
        .str.replace('"', "", regex=False)
        .str.strip()
    )

    # save file
    df.to_csv(
        file_path,
        index=False,
        encoding="utf-8-sig"
    )

    status = "SUCCESS"
    rows = len(df)

    if rows == 0:
        message = "No data yet"
    else:
        message = "File saved"

except Exception as e:

    status = "FAILED"
    rows = 0
    message = str(e)


# log row
log_row = pd.DataFrame([{
    "timestamp": datetime.now(),
    "dataset": "nse_block",
    "status": status,
    "rows": rows,
    "message": message
}])


# append log
if os.path.exists(log_file):

    old_log = pd.read_csv(log_file)

    log_df = pd.concat(
        [old_log, log_row],
        ignore_index=True
    )

else:
    log_df = log_row


log_df.to_csv(
    log_file,
    index=False
)

print(status)
print("Rows:", rows)
print(message)

StatementMeta(, 6e514e73-d0eb-4588-8654-a63fcf37a328, 3, Finished, Available, Finished, False)

SUCCESS
Rows: 29
File saved
